## 1.Cài đặt các thư viện và môi trường cần thiết

In [1]:
!pip install gensim
!pip install underthesea

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from underthesea import word_tokenize, pos_tag, sent_tokenize
import warnings
from gensim import corpora, models, similarities
import re

In [3]:
from google.colab import drive
drive.mount("/content/gdrive", force_remount=True)

%cd '/content/gdrive/MyDrive/Project'

Mounted at /content/gdrive
/content/gdrive/MyDrive/Project


## 2.Gensim

### 2.1 Đọc và xử lý dữ liệu

In [4]:
STOP_WORD_FILE = 'files/vietnamese-stopwords-D.txt'

In [5]:
with open(STOP_WORD_FILE, 'r', encoding='utf-8') as file:
    stop_words = file.read()

stop_words = stop_words.split('\n')

In [6]:
df= pd.read_csv('data/San_pham.csv')

In [7]:
df.head()

,ma_san_pham,ten_san_pham,gia_ban,gia_goc,phan_loai,mo_ta,diem_trung_binh
0,318900012,Nước Hoa Hồng Klairs Không Mùi Cho Da Nhạy Cảm...,209000,435000.0,2x180ml\n180ml\nKhông Mùi\nCó Mùi Hương,Nước Hoa Hồng Klairs Supple Preparation là dòn...,4.8
1,205100137,"Nước Tẩy Trang L'Oreal Tươi Mát Cho Da Dầu, Hỗ...",147000,229000.0,2x400ml\n95ml\n400ml\nLàm Sạch Sâu\nTươi Mát D...,Nước Tẩy Trang L'Oréal là dòng sản phẩm tẩy tr...,4.7
2,422208973,Sữa Rửa Mặt CeraVe Sạch Sâu Cho Da Thường Đến ...,343000,455000.0,88ml\n236ml\n473ml\nDa khô/Hỗn hợp khô\nDa dầu...,Sữa Rửa Mặt Cerave Sạch Sâu là sản phẩm sữa rử...,4.9
3,204900013,Kem Chống Nắng La Roche-Posay Kiểm Soát Dầu SP...,377000,560000.0,2x50ml\n50ml,Kem chống nắng giúp bảo vệ da khỏi tia UVB & U...,4.6
4,253900006,Kem Chống Nắng Skin1004 Cho Da Nhạy Cảm SPF 50...,210000,445000.0,20ml\n50ml,Kem Chống Nắng Skin1004 Cho Da Nhạy Cảm là sản...,4.6


#### 2.1.1 Xử lý dữ liệu

In [8]:
data_gen = df[['ma_san_pham', 'ten_san_pham', 'mo_ta']]
data_gen.head()

,ma_san_pham,ten_san_pham,mo_ta
0,318900012,Nước Hoa Hồng Klairs Không Mùi Cho Da Nhạy Cảm...,Nước Hoa Hồng Klairs Supple Preparation là dòn...
1,205100137,"Nước Tẩy Trang L'Oreal Tươi Mát Cho Da Dầu, Hỗ...",Nước Tẩy Trang L'Oréal là dòng sản phẩm tẩy tr...
2,422208973,Sữa Rửa Mặt CeraVe Sạch Sâu Cho Da Thường Đến ...,Sữa Rửa Mặt Cerave Sạch Sâu là sản phẩm sữa rử...
3,204900013,Kem Chống Nắng La Roche-Posay Kiểm Soát Dầu SP...,Kem chống nắng giúp bảo vệ da khỏi tia UVB & U...
4,253900006,Kem Chống Nắng Skin1004 Cho Da Nhạy Cảm SPF 50...,Kem Chống Nắng Skin1004 Cho Da Nhạy Cảm là sản...


In [9]:
data_gen['Content'] = data_gen['mo_ta'].apply(lambda x: ' '.join(x.split()))

<ipython-input-9-145073e6c9e6>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_gen['Content'] = data_gen['mo_ta'].apply(lambda x: ' '.join(x.split()))


In [10]:
data_gen.head()

,ma_san_pham,ten_san_pham,mo_ta,Content
0,318900012,Nước Hoa Hồng Klairs Không Mùi Cho Da Nhạy Cảm...,Nước Hoa Hồng Klairs Supple Preparation là dòn...,Nước Hoa Hồng Klairs Supple Preparation là dòn...
1,205100137,"Nước Tẩy Trang L'Oreal Tươi Mát Cho Da Dầu, Hỗ...",Nước Tẩy Trang L'Oréal là dòng sản phẩm tẩy tr...,Nước Tẩy Trang L'Oréal là dòng sản phẩm tẩy tr...
2,422208973,Sữa Rửa Mặt CeraVe Sạch Sâu Cho Da Thường Đến ...,Sữa Rửa Mặt Cerave Sạch Sâu là sản phẩm sữa rử...,Sữa Rửa Mặt Cerave Sạch Sâu là sản phẩm sữa rử...
3,204900013,Kem Chống Nắng La Roche-Posay Kiểm Soát Dầu SP...,Kem chống nắng giúp bảo vệ da khỏi tia UVB & U...,Kem chống nắng giúp bảo vệ da khỏi tia UVB & U...
4,253900006,Kem Chống Nắng Skin1004 Cho Da Nhạy Cảm SPF 50...,Kem Chống Nắng Skin1004 Cho Da Nhạy Cảm là sản...,Kem Chống Nắng Skin1004 Cho Da Nhạy Cảm là sản...


In [11]:
data_gen["Content_wt"]=data_gen["Content"].apply(lambda x: word_tokenize(x, format="text"))

<ipython-input-11-6bc2e37d4f3d>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_gen["Content_wt"]=data_gen["Content"].apply(lambda x: word_tokenize(x, format="text"))


In [12]:
data_gen[["Content", "Content_wt"]].head(2)

,Content,Content_wt
0,Nước Hoa Hồng Klairs Supple Preparation là dòn...,Nước Hoa_Hồng Klairs_Supple Preparation là dòn...
1,Nước Tẩy Trang L'Oréal là dòng sản phẩm tẩy tr...,Nước Tẩy_Trang L'Oréal là dòng sản_phẩm tẩy_tr...


In [13]:
content_gem = [[text for text in x.split()] for x in data_gen.Content_wt]
len(content_gem)

1200

#### 2.1.2 Xây dựng hàm xử lý tổng quát cho cột mô tả

In [14]:
import re

In [15]:
def preprocess_text(content_gem, stop_words):
  content_gem_re = [[re.sub('[0-9]+','', e) for e in text] for text in content_gem] # xem xét có cần bỏ các con số hay không
  content_gem_re = [[t.lower() for t in text if not t in ['', ' ', ',', '.', '...', '-',':', ';', '?', '%', '(', ')', '+', '/', "'", '&', '*', '"']] for text in  content_gem_re] # kiểm tra nội dung và đưa vào các ký tự đặc biệt
  content_gem_re = [[t for t in text if not t in stop_words] for text in content_gem_re] # stopword
  return content_gem_re

In [16]:
data_gen['content_gem_re'] = preprocess_text(content_gem, stop_words)
data_gen.head()

<ipython-input-16-ff8dc460d94d>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_gen['content_gem_re'] = preprocess_text(content_gem, stop_words)


,ma_san_pham,ten_san_pham,mo_ta,Content,Content_wt,content_gem_re
0,318900012,Nước Hoa Hồng Klairs Không Mùi Cho Da Nhạy Cảm...,Nước Hoa Hồng Klairs Supple Preparation là dòn...,Nước Hoa Hồng Klairs Supple Preparation là dòn...,Nước Hoa_Hồng Klairs_Supple Preparation là dòn...,"[hoa_hồng, klairs_supple, preparation, dòng, s..."
1,205100137,"Nước Tẩy Trang L'Oreal Tươi Mát Cho Da Dầu, Hỗ...",Nước Tẩy Trang L'Oréal là dòng sản phẩm tẩy tr...,Nước Tẩy Trang L'Oréal là dòng sản phẩm tẩy tr...,Nước Tẩy_Trang L'Oréal là dòng sản_phẩm tẩy_tr...,"[tẩy_trang, l'oréal, dòng, sản_phẩm, tẩy_trang..."
2,422208973,Sữa Rửa Mặt CeraVe Sạch Sâu Cho Da Thường Đến ...,Sữa Rửa Mặt Cerave Sạch Sâu là sản phẩm sữa rử...,Sữa Rửa Mặt Cerave Sạch Sâu là sản phẩm sữa rử...,Sữa Rửa Mặt_Cerave Sạch_Sâu là sản_phẩm sữa_rử...,"[sữa, rửa, mặt_cerave, sạch_sâu, sản_phẩm, sữa..."
3,204900013,Kem Chống Nắng La Roche-Posay Kiểm Soát Dầu SP...,Kem chống nắng giúp bảo vệ da khỏi tia UVB & U...,Kem chống nắng giúp bảo vệ da khỏi tia UVB & U...,Kem chống nắng giúp bảo_vệ da khỏi tia UVB & U...,"[kem, chống, nắng, giúp, bảo_vệ, da, tia, uvb,..."
4,253900006,Kem Chống Nắng Skin1004 Cho Da Nhạy Cảm SPF 50...,Kem Chống Nắng Skin1004 Cho Da Nhạy Cảm là sản...,Kem Chống Nắng Skin1004 Cho Da Nhạy Cảm là sản...,Kem Chống Nắng_Skin1004 Cho_Da Nhạy_Cảm là sản...,"[kem, chống, nắng_skin, cho_da, nhạy_cảm, sản_..."


In [17]:
dictionary = corpora.Dictionary(data_gen['content_gem_re'])
dictionary.token2id

{'chiết_xuất': 0,
 'cho_da': 1,
 'chuyên_biệt': 2,
 'cân_bằng': 3,
 'cơ_địa': 4,
 'cảm_giác': 5,
 'da': 6,
 'dòng': 7,
 'dưỡng_ẩm': 8,
 'dễ_chịu': 9,
 'facial_toner': 10,
 'giúp': 11,
 'hiệu_quả': 12,
 'hoa_hồng': 13,
 'hương': 14,
 'hương_thảo_mộc': 15,
 'klairs': 16,
 'klairs_supple': 17,
 'kích_ứng': 18,
 'kết_cấu': 19,
 'kết_hợp': 20,
 'làn': 21,
 'lưu_ý': 22,
 'lỏng': 23,
 'ml_nước': 24,
 'màu': 25,
 'mùi': 26,
 'nguồn_gốc': 27,
 'nhanh_chóng': 28,
 'nhạy_cảm': 29,
 'nhẹ': 30,
 'nhờn_dính': 31,
 'nước_hoa_hồng': 32,
 'nước_hoa_hồng_klairs': 33,
 'phân_biệt': 34,
 'preparation': 35,
 'preparation_facial': 36,
 'rõ_ràng': 37,
 'siêu_nhạy_cảm': 38,
 'skincare': 39,
 'suốt': 40,
 'sạch': 41,
 'sản_phẩm': 42,
 'thiết_kế': 43,
 'thành_phần': 44,
 'thương_hiệu': 45,
 'thảo_mộc': 46,
 'thấm': 47,
 'thẩm_thấu': 48,
 'thực_vật': 49,
 'toner': 50,
 'tác_dụng': 51,
 'tùy': 52,
 'tối_ưu': 53,
 'unscented_toner': 54,
 'việc_làm': 55,
 'vô_cùng': 56,
 'độ_ph': 57,
 'ẩm': 58,
 'bụi_bẩn': 59,
 'cô

In [18]:
feature_cnt = len(dictionary.token2id)
feature_cnt

5317

In [19]:
corpus = [dictionary.doc2bow(text) for text in data_gen['content_gem_re']]

In [20]:
# Sử dụng TF-IDF Model để xử lý corpus, trả về index
tfidf = models.TfidfModel(corpus)
# tính toán sự tương tự trong ma trận thưa thớt
index = similarities.SparseMatrixSimilarity(tfidf[corpus],
                                            num_features = feature_cnt)

In [21]:
df_1 = pd.DataFrame(index)
df_1

,0,1,2,3,4,5,6,7,8,9,...,1190,1191,1192,1193,1194,1195,1196,1197,1198,1199
0,1.000000,0.012517,0.014683,0.021371,0.040308,0.030247,0.001099,0.030247,0.005816,0.012517,...,0.022439,0.023625,0.016672,0.012517,0.020166,0.000050,0.003226,0.000198,0.041723,0.014476
1,0.012517,1.000000,0.023257,0.000714,0.025354,0.150569,0.017029,0.150569,0.010765,1.000000,...,0.014752,0.013113,0.030209,1.000000,0.015751,0.000081,0.085391,0.003991,0.008102,0.011779
2,0.014683,0.023257,1.000000,0.020988,0.009564,0.022163,0.010904,0.022163,0.013443,0.023257,...,0.010468,0.011453,0.014010,0.023257,0.005206,0.008699,0.028029,0.002189,0.033386,0.011151
3,0.021371,0.000714,0.020988,1.000000,0.218215,0.013781,0.102929,0.013781,0.152138,0.000714,...,0.048249,0.233674,0.002661,0.000714,0.176592,0.080237,0.007035,0.004612,0.012678,0.016882
4,0.040308,0.025354,0.009564,0.218215,1.000000,0.060819,0.149748,0.060819,0.175908,0.025354,...,0.032192,0.190110,0.022797,0.025354,0.245517,0.113607,0.004108,0.000076,0.059784,0.024898
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,0.000050,0.000081,0.008699,0.080237,0.113607,0.003416,0.100657,0.003416,0.164779,0.000081,...,0.004554,0.078077,0.002207,0.000081,0.123763,1.000000,0.000032,0.001350,0.002783,0.004204
1196,0.003226,0.085391,0.028029,0.007035,0.004108,0.088366,0.004787,0.088366,0.007659,0.085391,...,0.006271,0.002033,0.046341,0.085391,0.013469,0.000032,1.000000,0.029515,0.001024,0.007838
1197,0.000198,0.003991,0.002189,0.004612,0.000076,0.004005,0.002610,0.004005,0.002069,0.003991,...,0.000500,0.013296,0.001247,0.003991,0.000124,0.001350,0.029515,1.000000,0.000162,0.000056
1198,0.041723,0.008102,0.033386,0.012678,0.059784,0.009955,0.031463,0.009955,0.027769,0.008102,...,0.050972,0.026128,0.027867,0.008102,0.010273,0.002783,0.001024,0.000162,1.000000,0.012905


### 2.2 Xây dựng hàm tổng quát

#### 2.2.1 Xây dựng hàm bằng cách nhập ID sản phẩm

In [50]:
def get_recommendations_gen(product_id, data=data_gen, df=df, dictionary=dictionary, index=index, tfidf=tfidf):
    # Tìm index của sản phẩm dựa trên mã sản phẩm
    product_index = df[df['ma_san_pham'] == product_id].index[0]

    # Lấy nội dung của sản phẩm đầu vào
    view_content = data['content_gem_re'][product_index]

    # Chuyển nội dung thành vector BoW
    kw_vector = dictionary.doc2bow(view_content)

    # Tính toán mức độ tương tự giữa sản phẩm đầu vào và các sản phẩm khác
    sim = index[tfidf[kw_vector]]

    # Sắp xếp mức độ tương tự giảm dần
    sim_sort = sorted(enumerate(sim), key=lambda item: -item[1])

    # Khởi tạo danh sách các sản phẩm được gợi ý
    recommended_products = []

    # Lặp qua các sản phẩm tương tự
    for idx, sim_score in sim_sort[1:]:
        # Kiểm tra nếu điểm trung bình của sản phẩm này >= 3
        if df.iloc[idx]['diem_trung_binh'] >= 3:
            recommended_products.append(idx)
        # Nếu đã có đủ 3 sản phẩm, thoát vòng lặp
        if len(recommended_products) == 5:
            break

    # Trả về thông tin của 3 sản phẩm được gợi ý từ DataFrame gốc
    return df[df['ma_san_pham'] == product_id][['ma_san_pham', 'ten_san_pham', 'gia_ban', 'gia_goc', 'mo_ta', 'diem_trung_binh']], df.iloc[recommended_products][['ma_san_pham', 'ten_san_pham', 'gia_ban', 'gia_goc', 'mo_ta', 'diem_trung_binh']]

In [51]:
sp, rec_sp = get_recommendations_gen(422207117)

In [52]:
sp

,ma_san_pham,ten_san_pham,gia_ban,gia_goc,mo_ta,diem_trung_binh
1199,422207117,Lotion Curél Dưỡng Ẩm Chuyên Sâu Cho Da Lão Hó...,494000,641000.0,Lotion Curél Dưỡng Ẩm Chuyên Sâu Cho Da Lão Hó...,5.0


In [53]:
rec_sp

,ma_san_pham,ten_san_pham,gia_ban,gia_goc,mo_ta,diem_trung_binh
784,331300019,Gel Dưỡng Da Curél Dành Cho Da Dầu 120ml,457000,610000.0,Curél Sebum Trouble Care Sebum Care Gel là sản...,5.0
973,248700040,"Lotion Naris Cosmetics Dưỡng Ẩm Da, Ngăn Ngừa ...",396000,495000.0,"Lotion Naris Cosmetics Dưỡng Ẩm Da, Ngăn Ngừa ...",5.0
953,331300001,Gel Tẩy Trang Curél Cấp Ẩm Chuyên Sâu 130g,248000,339000.0,Gel Tẩy Trang Curél Cấp Ẩm Chuyên Sâu 130g là ...,5.0
1169,331300006,Sữa Dưỡng Da Curél Cấp Ẩm Chuyên Sâu 120ml,487000,610000.0,Sữa Dưỡng Da Curél Intensive Moisture Care Moi...,5.0
415,232500001,Lotion Meishoku Bigansui Ngăn Ngừa Mụn 90ml,178000,280000.0,Meishoku Bigansui Medicated Skin Lotion là dòn...,4.6


#### 2.2.2 Xây dựng hàm bằng cách nhập từ khóa

In [68]:
def get_recommendations_by_keyword_gen(input_keyword, data=data_gen, df=df, dictionary=dictionary, index=index, tfidf=tfidf):
    """
    Đề xuất sản phẩm dựa trên một chuỗi ký tự bất kỳ do người dùng nhập vào.

    Args:
    - input_keyword (str): Chuỗi ký tự người dùng nhập.
    - data (pd.DataFrame): Dữ liệu sản phẩm với cột 'content_gem_re'.
    - df (pd.DataFrame): Dữ liệu sản phẩm gốc chứa thông tin sản phẩm.
    - dictionary: Mô hình từ điển (Doc2Bow) cho nội dung sản phẩm.
    - index: Chỉ mục mô hình tương tự.
    - tfidf: Mô hình TF-IDF.

    Returns:
    - pd.DataFrame: Thông tin của 5 sản phẩm được đề xuất bao gồm mã sản phẩm, tên sản phẩm, và điểm trung bình.
    """
    # Tìm các sản phẩm có chứa chuỗi ký tự trong tên sản phẩm
    matched_products = df[df['mo_ta'].str.contains(input_keyword, case=False, na=False)]

    if matched_products.empty:
        print("Không tìm thấy sản phẩm nào khớp với từ khóa.")
        return pd.DataFrame(columns=['ma_san_pham', 'ten_san_pham', 'gia_ban', 'gia_goc', 'mo_ta', 'diem_trung_binh'])

    # Chọn sản phẩm đầu tiên từ các sản phẩm khớp để làm đầu vào
    product_index = matched_products.index[0]

    # Lấy nội dung của sản phẩm đầu vào
    view_content = data['content_gem_re'][product_index]

    # Chuyển nội dung thành vector BoW
    kw_vector = dictionary.doc2bow(view_content)

    # Tính toán mức độ tương tự giữa sản phẩm đầu vào và các sản phẩm khác
    sim = index[tfidf[kw_vector]]

    # Sắp xếp mức độ tương tự giảm dần
    sim_sort = sorted(enumerate(sim), key=lambda item: -item[1])

    # Khởi tạo danh sách các sản phẩm được gợi ý
    recommended_products = []

    # Lặp qua các sản phẩm tương tự
    for idx, sim_score in sim_sort[1:]:
        # Kiểm tra nếu điểm trung bình của sản phẩm này >= 3
        if df.iloc[idx]['diem_trung_binh'] >= 3:
            recommended_products.append(idx)
        # Nếu đã có đủ 5 sản phẩm, thoát vòng lặp
        if len(recommended_products) == 5:
            break

    # Trả về thông tin của 5 sản phẩm được gợi ý từ DataFrame gốc
    return df.iloc[recommended_products][['ma_san_pham', 'ten_san_pham', 'gia_ban', 'gia_goc', 'mo_ta', 'diem_trung_binh']]


In [66]:
keyword = input("Nhập từ khóa để tìm kiếm sản phẩm: ")

get_recommendations_by_keyword_gen(keyword)

Nhập từ khóa để tìm kiếm sản phẩm: da đầu


,ma_san_pham,ten_san_pham,gia_ban,gia_goc,mo_ta,diem_trung_binh
22,204900024,Nước Tẩy Trang La Roche-Posay Dành Cho Da Nhạy...,379000,525000.0,Nước Tẩy Trang La Roche-Posay Dành Cho Da Nhạy...,4.8
358,204900005,[Mini] Nước Tẩy Trang La Roche-Posay Dành Cho ...,69000,155000.0,Nước Tẩy Trang La Roche-Posay Dành Cho Da Nhạy...,4.8
981,422216603,Combo 2 Nước Tẩy Trang La Roche-Posay Dành Cho...,711000,990000.0,Nước Tẩy Trang La Roche-Posay Dành Cho Da Nhạy...,4.8
731,204900009,Nước Tẩy Trang La Roche-Posay Dành Cho Da Nhạy...,318000,425000.0,Nước Tẩy Trang La Roche-Posay Dành Cho Da Nhạy...,4.8
359,204900120,"Combo La Roche-Posay Làm Sạch Sâu Cho Da Dầu, ...",349000,450000.0,"Combo Làm Sạch Da La Roche-Posay Cho Da Dầu, N...",4.0


## 3.Cosine Similarity

### 3.1 Kiểm tra dữ liệu và xử lý dữ liệu

In [30]:
data_cos = df[['ma_san_pham', 'ten_san_pham', 'mo_ta']]
data_cos.head()

,ma_san_pham,ten_san_pham,mo_ta
0,318900012,Nước Hoa Hồng Klairs Không Mùi Cho Da Nhạy Cảm...,Nước Hoa Hồng Klairs Supple Preparation là dòn...
1,205100137,"Nước Tẩy Trang L'Oreal Tươi Mát Cho Da Dầu, Hỗ...",Nước Tẩy Trang L'Oréal là dòng sản phẩm tẩy tr...
2,422208973,Sữa Rửa Mặt CeraVe Sạch Sâu Cho Da Thường Đến ...,Sữa Rửa Mặt Cerave Sạch Sâu là sản phẩm sữa rử...
3,204900013,Kem Chống Nắng La Roche-Posay Kiểm Soát Dầu SP...,Kem chống nắng giúp bảo vệ da khỏi tia UVB & U...
4,253900006,Kem Chống Nắng Skin1004 Cho Da Nhạy Cảm SPF 50...,Kem Chống Nắng Skin1004 Cho Da Nhạy Cảm là sản...


#### 3.1.1 Xử lý dữ liệu

In [32]:
data_cos['Content'] = data_cos['mo_ta'].apply(lambda x: ' '.join(x.split()))

<ipython-input-32-ecf32ab41bf4>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_cos['Content'] = data_cos['mo_ta'].apply(lambda x: ' '.join(x.split()))


In [33]:
data_cos[['ma_san_pham','mo_ta', 'Content']].head()

,ma_san_pham,mo_ta,Content
0,318900012,Nước Hoa Hồng Klairs Supple Preparation là dòn...,Nước Hoa Hồng Klairs Supple Preparation là dòn...
1,205100137,Nước Tẩy Trang L'Oréal là dòng sản phẩm tẩy tr...,Nước Tẩy Trang L'Oréal là dòng sản phẩm tẩy tr...
2,422208973,Sữa Rửa Mặt Cerave Sạch Sâu là sản phẩm sữa rử...,Sữa Rửa Mặt Cerave Sạch Sâu là sản phẩm sữa rử...
3,204900013,Kem chống nắng giúp bảo vệ da khỏi tia UVB & U...,Kem chống nắng giúp bảo vệ da khỏi tia UVB & U...
4,253900006,Kem Chống Nắng Skin1004 Cho Da Nhạy Cảm là sản...,Kem Chống Nắng Skin1004 Cho Da Nhạy Cảm là sản...


In [34]:
data_cos["Content_wt"] = data_cos["Content"].apply(lambda x: word_tokenize(x, format="text"))

<ipython-input-34-a702b73d4a26>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_cos["Content_wt"] = data_cos["Content"].apply(lambda x: word_tokenize(x, format="text"))


In [36]:
data_cos['content_token'] = [[text for text in x.split()] for x in data_cos.Content_wt]

<ipython-input-36-182d74c864eb>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_cos['content_token'] = [[text for text in x.split()] for x in data_cos.Content_wt]


In [37]:
data_cos.head()

,ma_san_pham,ten_san_pham,mo_ta,Content,Content_wt,content_token
0,318900012,Nước Hoa Hồng Klairs Không Mùi Cho Da Nhạy Cảm...,Nước Hoa Hồng Klairs Supple Preparation là dòn...,Nước Hoa Hồng Klairs Supple Preparation là dòn...,Nước Hoa_Hồng Klairs_Supple Preparation là dòn...,"[Nước, Hoa_Hồng, Klairs_Supple, Preparation, l..."
1,205100137,"Nước Tẩy Trang L'Oreal Tươi Mát Cho Da Dầu, Hỗ...",Nước Tẩy Trang L'Oréal là dòng sản phẩm tẩy tr...,Nước Tẩy Trang L'Oréal là dòng sản phẩm tẩy tr...,Nước Tẩy_Trang L'Oréal là dòng sản_phẩm tẩy_tr...,"[Nước, Tẩy_Trang, L'Oréal, là, dòng, sản_phẩm,..."
2,422208973,Sữa Rửa Mặt CeraVe Sạch Sâu Cho Da Thường Đến ...,Sữa Rửa Mặt Cerave Sạch Sâu là sản phẩm sữa rử...,Sữa Rửa Mặt Cerave Sạch Sâu là sản phẩm sữa rử...,Sữa Rửa Mặt_Cerave Sạch_Sâu là sản_phẩm sữa_rử...,"[Sữa, Rửa, Mặt_Cerave, Sạch_Sâu, là, sản_phẩm,..."
3,204900013,Kem Chống Nắng La Roche-Posay Kiểm Soát Dầu SP...,Kem chống nắng giúp bảo vệ da khỏi tia UVB & U...,Kem chống nắng giúp bảo vệ da khỏi tia UVB & U...,Kem chống nắng giúp bảo_vệ da khỏi tia UVB & U...,"[Kem, chống, nắng, giúp, bảo_vệ, da, khỏi, tia..."
4,253900006,Kem Chống Nắng Skin1004 Cho Da Nhạy Cảm SPF 50...,Kem Chống Nắng Skin1004 Cho Da Nhạy Cảm là sản...,Kem Chống Nắng Skin1004 Cho Da Nhạy Cảm là sản...,Kem Chống Nắng_Skin1004 Cho_Da Nhạy_Cảm là sản...,"[Kem, Chống, Nắng_Skin1004, Cho_Da, Nhạy_Cảm, ..."


In [38]:
def preprocess_content(content, stop_words):
    # Bỏ các con số
    content_pre = [[re.sub('[0-9]+','', e) for e in text] for text in content]

    # Chuyển thành chữ thường và loại bỏ các ký tự đặc biệt
    special_characters = ['', ' ', ',', '.', '...', '-', ':', ';', '?', '%', '(', ')', '+', '/', "'", '&', '*', '"']
    content_pre = [[t.lower() for t in text if not t in special_characters] for text in content_pre]

    # Loại bỏ stop words
    content_pre = [[t for t in text if not t in stop_words] for text in content_pre]

    return content_pre

In [39]:
data_cos['processed_content'] = data_cos['content_token'].apply(lambda x: preprocess_content([x], stop_words)[0])

In [40]:
data_cos['processed_content'] = data_cos['processed_content'].apply(lambda x: ' '.join(x))

#### 3.1.2 Chuẩn hóa vecto và đưa ra ma trận tương đồng

In [42]:
vectorizer = TfidfVectorizer(analyzer='word')
tfidf_matrix = vectorizer.fit_transform(data_cos['processed_content'])

In [43]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [44]:
df_sim = pd.DataFrame(cosine_sim)
df_sim

,0,1,2,3,4,5,6,7,8,9,...,1190,1191,1192,1193,1194,1195,1196,1197,1198,1199
0,1.000000,0.048580,0.040296,0.044963,0.081155,0.062260,0.017626,0.062260,0.015482,0.048580,...,0.056147,0.048118,0.048433,0.048580,0.038886,0.008002,0.015827,0.006610,0.065180,0.048656
1,0.048580,1.000000,0.060409,0.034739,0.072672,0.231735,0.059034,0.231735,0.048652,1.000000,...,0.061097,0.043361,0.074765,1.000000,0.036320,0.015045,0.137611,0.017461,0.034954,0.059990
2,0.040296,0.060409,1.000000,0.060241,0.040822,0.051340,0.034904,0.051340,0.027530,0.060409,...,0.041019,0.032739,0.041379,0.060409,0.017022,0.017004,0.052315,0.009618,0.059358,0.045971
3,0.044963,0.034739,0.060241,1.000000,0.313781,0.042476,0.168932,0.042476,0.197102,0.034739,...,0.090806,0.317364,0.034850,0.034739,0.262514,0.129863,0.023357,0.012304,0.076468,0.058738
4,0.081155,0.072672,0.040822,0.313781,1.000000,0.101322,0.233126,0.101322,0.236399,0.072672,...,0.081283,0.301190,0.059691,0.072672,0.354232,0.175840,0.021850,0.009439,0.108047,0.076917
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,0.008002,0.015045,0.017004,0.129863,0.175840,0.014403,0.167418,0.014403,0.220554,0.015045,...,0.018477,0.143030,0.012758,0.015045,0.196014,1.000000,0.005766,0.005872,0.010821,0.020851
1196,0.015827,0.137611,0.052315,0.023357,0.021850,0.137770,0.021801,0.137770,0.015769,0.137611,...,0.023657,0.011858,0.064981,0.137611,0.025903,0.005766,1.000000,0.044814,0.010864,0.027882
1197,0.006610,0.017461,0.009618,0.012304,0.009439,0.013391,0.011634,0.013391,0.005748,0.017461,...,0.010258,0.019596,0.008203,0.017461,0.003436,0.005872,0.044814,1.000000,0.005090,0.009441
1198,0.065180,0.034954,0.059358,0.076468,0.108047,0.026521,0.054101,0.026521,0.042469,0.034954,...,0.081773,0.053659,0.032513,0.034954,0.027099,0.010821,0.010864,0.005090,1.000000,0.036015


### 3.2 Xây dựng hàm tổng quát

#### 3.2.1 Xây dựng hàm bằng cách nhập ID sản phẩm

In [55]:
def get_recommendations_cos(sp_id, cosine_sim=cosine_sim, nums=5):
    # Lấy chỉ mục của sản phẩm đầu vào
    idx = df.index[df['ma_san_pham'] == sp_id][0]

    # Tính toán độ tương đồng
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:]  # Bỏ sản phẩm đầu tiên (chính sản phẩm đó)

    # Lọc các sản phẩm tương tự nhất dựa trên nums
    sp_indices = [i[0] for i in sim_scores]

    # Lấy thông tin sản phẩm dựa trên chỉ số
    recommendations = df[['ma_san_pham', 'ten_san_pham', 'gia_ban', 'gia_goc', 'mo_ta', 'diem_trung_binh']].iloc[sp_indices]

    # Áp dụng điều kiện: diem_trung_binh >= 3
    recommendations = recommendations[recommendations['diem_trung_binh'] >= 3]

    # Lấy nums sản phẩm đầu tiên (nếu có đủ)
    return df[df['ma_san_pham'] == sp_id][['ma_san_pham', 'ten_san_pham', 'gia_ban', 'gia_goc', 'mo_ta', 'diem_trung_binh']], recommendations.head(nums)

In [56]:
sp, rec_sp = get_recommendations_cos(422207117)

In [57]:
sp

,ma_san_pham,ten_san_pham,gia_ban,gia_goc,mo_ta,diem_trung_binh
1199,422207117,Lotion Curél Dưỡng Ẩm Chuyên Sâu Cho Da Lão Hó...,494000,641000.0,Lotion Curél Dưỡng Ẩm Chuyên Sâu Cho Da Lão Hó...,5.0


In [58]:
rec_sp

,ma_san_pham,ten_san_pham,gia_ban,gia_goc,mo_ta,diem_trung_binh
973,248700040,"Lotion Naris Cosmetics Dưỡng Ẩm Da, Ngăn Ngừa ...",396000,495000.0,"Lotion Naris Cosmetics Dưỡng Ẩm Da, Ngăn Ngừa ...",5.0
784,331300019,Gel Dưỡng Da Curél Dành Cho Da Dầu 120ml,457000,610000.0,Curél Sebum Trouble Care Sebum Care Gel là sản...,5.0
214,224100016,Kem Dưỡng Hada Labo Chuyên Biệt Cải Thiện Da L...,224000,295000.0,Kem Dưỡng Hada Labo Chuyên Biệt Cải Thiện Da L...,4.4
953,331300001,Gel Tẩy Trang Curél Cấp Ẩm Chuyên Sâu 130g,248000,339000.0,Gel Tẩy Trang Curél Cấp Ẩm Chuyên Sâu 130g là ...,5.0
963,248700073,Nước Dưỡng Da Naris Nature Tẩy Tế Bào Chết 2in...,464000,485000.0,Nước Dưỡng & Tẩy Tế Bào Chết 2 Trong 1 Naris N...,4.4


In [67]:
# Gợi ý sản phẩm dựa trên từ khóa mà người dùng nhập.
def get_recommendations_by_keyword_cos(keyword, df, cosine_sim=cosine_sim, nums=3):
    """
    Parameters:
    - keyword: Chuỗi người dùng nhập để tìm kiếm sản phẩm.
    - df: DataFrame chứa thông tin sản phẩm.
    - cosine_sim: Ma trận cosine similarity.
    - nums: Số lượng sản phẩm gợi ý.

    Returns:
    - recommendations: DataFrame chứa danh sách sản phẩm gợi ý.
    """
    # Tìm các sản phẩm có tên chứa từ khóa (không phân biệt hoa thường)
    matches = df[df['mo_ta'].str.contains(keyword, case=False, na=False)]

    if matches.empty:
        return f"Không tìm thấy sản phẩm nào khớp với từ khóa: '{keyword}'"

    # Nếu tìm thấy nhiều sản phẩm, chỉ lấy sản phẩm đầu tiên làm cơ sở tính toán
    sp_id = matches.iloc[0]['ma_san_pham']
    sp, rec_sp = get_recommendations_cos(sp_id, cosine_sim=cosine_sim, nums=nums)
    recommendations = pd.concat([sp, rec_sp])
    return recommendations.head(5)

In [65]:
keyword = input("Nhập từ khóa để tìm kiếm sản phẩm: ")

# Lấy gợi ý sản phẩm
rec = get_recommendations_by_keyword_cos(keyword, df, cosine_sim, nums=5)
rec

Nhập từ khóa để tìm kiếm sản phẩm: da đầu


,ma_san_pham,ten_san_pham,gia_ban,gia_goc,mo_ta,diem_trung_binh
580,204900119,Combo La Roche-Posay Nước Tẩy Trang & Làm Dịu ...,429000,580000.0,Combo La Roche-Posay Nước Tẩy Trang Cho Da Nhạ...,5.0
22,204900024,Nước Tẩy Trang La Roche-Posay Dành Cho Da Nhạy...,379000,525000.0,Nước Tẩy Trang La Roche-Posay Dành Cho Da Nhạy...,4.8
358,204900005,[Mini] Nước Tẩy Trang La Roche-Posay Dành Cho ...,69000,155000.0,Nước Tẩy Trang La Roche-Posay Dành Cho Da Nhạy...,4.8
981,422216603,Combo 2 Nước Tẩy Trang La Roche-Posay Dành Cho...,711000,990000.0,Nước Tẩy Trang La Roche-Posay Dành Cho Da Nhạy...,4.8
359,204900120,"Combo La Roche-Posay Làm Sạch Sâu Cho Da Dầu, ...",349000,450000.0,"Combo Làm Sạch Da La Roche-Posay Cho Da Dầu, N...",4.0


## 4.So sánh

### 4.1 Đề xuất bằng ID sản phẩm

In [82]:
import time

def combined_recommendations(product_id,
                                data_gen=data_gen, df=df, dictionary=dictionary, index=index, tfidf=tfidf,
                                cosine_sim=cosine_sim, nums=5):
    # Đo thời gian thực thi hàm get_recommendations_gen
    start_gen = time.time()
    gen_input, gen_recommendations = get_recommendations_gen(
        product_id, data=data_gen, df=df, dictionary=dictionary, index=index, tfidf=tfidf
    )
    gen_time = time.time() - start_gen

    # Đo thời gian thực thi hàm get_recommendations_cos
    start_cos = time.time()
    cos_input, cos_recommendations = get_recommendations_cos(
        product_id, cosine_sim=cosine_sim, nums=nums
    )
    cos_time = time.time() - start_cos

    # Tạo DataFrame từ kết quả gensim
    gen_df = gen_recommendations.copy()
    gen_df['method'] = 'gensim'
    gen_df['execution_time'] = gen_time

    # Tạo DataFrame từ kết quả cosine
    cos_df = cos_recommendations.copy()
    cos_df['method'] = 'cosine'
    cos_df['execution_time'] = cos_time

    # Ghép hai DataFrame lại với nhau
    combined_df = pd.concat([gen_df, cos_df], ignore_index=True)

    return combined_df

In [78]:
pro_id = int(input("Nhập mã sản phẩm: "))
rs = combined_recommendations(pro_id)
gen = rs[rs['method'] == 'gensim']
cos = rs[rs['method'] == 'cosine']

Nhập mã sản phẩm: 422207117


In [79]:
df[df['ma_san_pham']==pro_id]

,ma_san_pham,ten_san_pham,gia_ban,gia_goc,phan_loai,mo_ta,diem_trung_binh
1199,422207117,Lotion Curél Dưỡng Ẩm Chuyên Sâu Cho Da Lão Hó...,494000,641000.0,NaN,Lotion Curél Dưỡng Ẩm Chuyên Sâu Cho Da Lão Hó...,5.0


In [80]:
gen

,ma_san_pham,ten_san_pham,gia_ban,gia_goc,mo_ta,diem_trung_binh,method,execution_time
0,331300019,Gel Dưỡng Da Curél Dành Cho Da Dầu 120ml,457000,610000.0,Curél Sebum Trouble Care Sebum Care Gel là sản...,5.0,gensim,0.008509
1,248700040,"Lotion Naris Cosmetics Dưỡng Ẩm Da, Ngăn Ngừa ...",396000,495000.0,"Lotion Naris Cosmetics Dưỡng Ẩm Da, Ngăn Ngừa ...",5.0,gensim,0.008509
2,331300001,Gel Tẩy Trang Curél Cấp Ẩm Chuyên Sâu 130g,248000,339000.0,Gel Tẩy Trang Curél Cấp Ẩm Chuyên Sâu 130g là ...,5.0,gensim,0.008509
3,331300006,Sữa Dưỡng Da Curél Cấp Ẩm Chuyên Sâu 120ml,487000,610000.0,Sữa Dưỡng Da Curél Intensive Moisture Care Moi...,5.0,gensim,0.008509
4,232500001,Lotion Meishoku Bigansui Ngăn Ngừa Mụn 90ml,178000,280000.0,Meishoku Bigansui Medicated Skin Lotion là dòn...,4.6,gensim,0.008509


In [81]:
cos

,ma_san_pham,ten_san_pham,gia_ban,gia_goc,mo_ta,diem_trung_binh,method,execution_time
5,248700040,"Lotion Naris Cosmetics Dưỡng Ẩm Da, Ngăn Ngừa ...",396000,495000.0,"Lotion Naris Cosmetics Dưỡng Ẩm Da, Ngăn Ngừa ...",5.0,cosine,0.00386
6,331300019,Gel Dưỡng Da Curél Dành Cho Da Dầu 120ml,457000,610000.0,Curél Sebum Trouble Care Sebum Care Gel là sản...,5.0,cosine,0.00386
7,224100016,Kem Dưỡng Hada Labo Chuyên Biệt Cải Thiện Da L...,224000,295000.0,Kem Dưỡng Hada Labo Chuyên Biệt Cải Thiện Da L...,4.4,cosine,0.00386
8,331300001,Gel Tẩy Trang Curél Cấp Ẩm Chuyên Sâu 130g,248000,339000.0,Gel Tẩy Trang Curél Cấp Ẩm Chuyên Sâu 130g là ...,5.0,cosine,0.00386
9,248700073,Nước Dưỡng Da Naris Nature Tẩy Tế Bào Chết 2in...,464000,485000.0,Nước Dưỡng & Tẩy Tế Bào Chết 2 Trong 1 Naris N...,4.4,cosine,0.00386


Nhận xét:
1. Chất lượng gợi ý:

  - Mô hình gensim:
    - Các sản phẩm gợi ý có điểm trung bình cao (từ 4.6 đến 5.0), thể hiện sự phù hợp cao với sản phẩm đầu vào.
    - Danh sách các sản phẩm gợi ý có tính liên quan cao, đa số cùng thuộc nhóm sản phẩm dưỡng da.
  - Mô hình cosine:
    - Các sản phẩm được gợi ý cũng có điểm trung bình cao, tuy nhiên, độ chính xác hơi kém hơn so với gensim (xuất hiện một số sản phẩm có điểm trung bình 4.4).
    - Gợi ý bao gồm một số sản phẩm có mức độ liên quan thấp hơn.
2. Tốc độ thực thi:

  - gensim: Mất khoảng 0.008509 giây để thực hiện.
  - cosine: Nhanh hơn, chỉ mất 0.003386 giây.
- -> cosine có ưu thế về tốc độ khi xử lý truy vấn.
3. Độ đa dạng trong kết quả:

  - gensim: Gợi ý tập trung vào các sản phẩm cùng thương hiệu hoặc cùng nhóm (dưỡng ẩm, làm sạch).
  - cosine: Đưa ra một số sản phẩm có mức đa dạng hơn nhưng có thể kém liên quan đến sản phẩm đầu vào.
4. Lựa chọn mô hình:
     - Vì kết quả chính xác hơn dựa trên phân tích nội dung (sử dụng TF-IDF và gensim) và tốc độ không quá đáng kể. -> Lựa chọn gensim.

### 4.2 Đề xuất bằng từ khóa do người dùng nhập

In [83]:
def combined_recommendations_by_keyword(keyword,
                                        data_gen=data_gen, df=df, dictionary=dictionary, index=index, tfidf=tfidf,
                                        cosine_sim=cosine_sim, nums=5):
    """
    Tổng hợp gợi ý sản phẩm từ hai phương pháp (gensim và cosine similarity) dựa trên từ khóa.

    Args:
    - keyword (str): Chuỗi ký tự người dùng nhập.
    - data_gen, df, dictionary, index, tfidf: Dữ liệu và mô hình cho phương pháp gensim.
    - cosine_sim: Ma trận cosine similarity.
    - nums (int): Số lượng sản phẩm gợi ý tối đa cho mỗi phương pháp.

    Returns:
    - pd.DataFrame: Kết quả gợi ý sản phẩm từ hai phương pháp và thời gian tính toán.
    """
    # Đo thời gian thực thi hàm gensim
    start_gen = time.time()
    try:
        gen_recommendations = get_recommendations_by_keyword_gen(
            keyword, data=data_gen, df=df, dictionary=dictionary, index=index, tfidf=tfidf
        )
        gen_time = time.time() - start_gen
    except Exception as e:
        gen_recommendations = pd.DataFrame(columns=['ma_san_pham', 'ten_san_pham', 'gia_ban', 'gia_goc', 'mo_ta', 'diem_trung_binh'])
        gen_time = -1  # Đánh dấu lỗi

    # Đo thời gian thực thi hàm cosine similarity
    start_cos = time.time()
    try:
        cos_recommendations = get_recommendations_by_keyword_cos(keyword, df, cosine_sim=cosine_sim, nums=nums)
        cos_time = time.time() - start_cos
    except Exception as e:
        cos_recommendations = pd.DataFrame(columns=['ma_san_pham', 'ten_san_pham', 'gia_ban', 'gia_goc', 'mo_ta', 'diem_trung_binh'])
        cos_time = -1  # Đánh dấu lỗi

    # Thêm cột phương pháp và thời gian tính toán
    gen_recommendations['method'] = 'gensim'
    gen_recommendations['execution_time'] = gen_time

    if isinstance(cos_recommendations, pd.DataFrame):
        cos_recommendations['method'] = 'cosine'
        cos_recommendations['execution_time'] = cos_time
    else:
        # Nếu không tìm thấy sản phẩm, tạo DataFrame rỗng
        cos_recommendations = pd.DataFrame(columns=['ma_san_pham', 'ten_san_pham', 'gia_ban', 'gia_goc', 'mo_ta', 'diem_trung_binh', 'method', 'execution_time'])

    # Gộp kết quả từ hai phương pháp
    combined_df = pd.concat([gen_recommendations, cos_recommendations], ignore_index=True)

    return combined_df

In [91]:
keyword = "da đầu"
result_df = combined_recommendations_by_keyword(keyword)

gen = result_df[result_df['method'] == 'gensim']
cos = result_df[result_df['method'] == 'cosine']

In [92]:
gen

,ma_san_pham,ten_san_pham,gia_ban,gia_goc,mo_ta,diem_trung_binh,method,execution_time
0,204900024,Nước Tẩy Trang La Roche-Posay Dành Cho Da Nhạy...,379000,525000.0,Nước Tẩy Trang La Roche-Posay Dành Cho Da Nhạy...,4.8,gensim,0.029861
1,204900005,[Mini] Nước Tẩy Trang La Roche-Posay Dành Cho ...,69000,155000.0,Nước Tẩy Trang La Roche-Posay Dành Cho Da Nhạy...,4.8,gensim,0.029861
2,422216603,Combo 2 Nước Tẩy Trang La Roche-Posay Dành Cho...,711000,990000.0,Nước Tẩy Trang La Roche-Posay Dành Cho Da Nhạy...,4.8,gensim,0.029861
3,204900009,Nước Tẩy Trang La Roche-Posay Dành Cho Da Nhạy...,318000,425000.0,Nước Tẩy Trang La Roche-Posay Dành Cho Da Nhạy...,4.8,gensim,0.029861
4,204900120,"Combo La Roche-Posay Làm Sạch Sâu Cho Da Dầu, ...",349000,450000.0,"Combo Làm Sạch Da La Roche-Posay Cho Da Dầu, N...",4.0,gensim,0.029861


In [93]:
cos

,ma_san_pham,ten_san_pham,gia_ban,gia_goc,mo_ta,diem_trung_binh,method,execution_time
5,204900119,Combo La Roche-Posay Nước Tẩy Trang & Làm Dịu ...,429000,580000.0,Combo La Roche-Posay Nước Tẩy Trang Cho Da Nhạ...,5.0,cosine,0.024976
6,204900024,Nước Tẩy Trang La Roche-Posay Dành Cho Da Nhạy...,379000,525000.0,Nước Tẩy Trang La Roche-Posay Dành Cho Da Nhạy...,4.8,cosine,0.024976
7,204900005,[Mini] Nước Tẩy Trang La Roche-Posay Dành Cho ...,69000,155000.0,Nước Tẩy Trang La Roche-Posay Dành Cho Da Nhạy...,4.8,cosine,0.024976
8,422216603,Combo 2 Nước Tẩy Trang La Roche-Posay Dành Cho...,711000,990000.0,Nước Tẩy Trang La Roche-Posay Dành Cho Da Nhạy...,4.8,cosine,0.024976
9,204900120,"Combo La Roche-Posay Làm Sạch Sâu Cho Da Dầu, ...",349000,450000.0,"Combo Làm Sạch Da La Roche-Posay Cho Da Dầu, N...",4.0,cosine,0.024976


Nhận xét:
  1. Chất lượng gợi ý:

    - Cả hai mô hình gensim và cosine đều gợi ý các sản phẩm liên quan đến từ khóa, với mô tả và thông tin khá tương tự nhau.
    - Điểm trung bình của các sản phẩm được gợi ý trong cả hai phương pháp đều đạt từ 4.0 trở lên, cho thấy cả hai mô hình đều đưa ra gợi ý chất lượng.
  2. Thời gian thực thi:

    - gensim có thời gian thực thi trung bình khoảng 0.029861 giây.
    - cosine có thời gian thực thi trung bình khoảng 0.024976 giây.
  - -> Mô hình cosine nhanh hơn so với gensim trong trường hợp này, nhưng sự khác biệt là không đáng kể (khoảng 0.005 giây).
  3. Độ đa dạng trong kết quả:

    - gensim có vẻ tập trung vào việc gợi ý các sản phẩm rất gần với đầu vào (các mô tả sản phẩm giống hệt nhau).
    - cosine gợi ý nhiều sản phẩm có liên quan nhưng có sự đa dạng hơn một chút về tên sản phẩm và điểm trung bình.
  4. Lựa chọn mô hình:
     - Vì kết quả chính xác hơn dựa trên phân tích nội dung (sử dụng TF-IDF và gensim) và tốc độ không quá đáng kể. -> Lựa chọn gensim.